In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "data/OMXS22_raw_features.csv",
    parse_dates=["Date"],
    index_col="Date"
)
start_test_point = df.index[-510*22]
print(start_test_point)
end_test_point = df.index[-10*22]
print(end_test_point)
all_tickers = df['Ticker'].unique()
print(all_tickers)


feature_cols = ["Close","Volume",
                "MarketCap","Weight",
                "Return","LogReturn","WeightedRet",
                "SMA20","EMA20","RSI14","ReturnVola20"]

chosen_ticker = "KINV-B.ST"

import numpy, torch, scipy
print("NumPy :", numpy.__version__)   # → 1.26.4
print("Torch :", torch.__version__)   # → 2.2.2
print("SciPy :", scipy.__version__)   # → 1.15.2

In [ ]:

window_step_length = 5
window_size = 40
num_test_points = 500

horizon = 20 #3 is good
target_idx = 0



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
trading_cost = 0.005

from allocation_functions.allocation_functions import allocate_from_weights

def plot_allocations(alloc, tickers, dates, title):
    n_assets, T = alloc.shape
    # Stack allocations
    cumsum = np.cumsum(alloc, axis=0)
    base_cmap = plt.get_cmap('tab20c')
    extra_cmap = plt.get_cmap('tab20b')
    colors = []
    for i in range(n_assets):
        if i < 10:
            colors.append(base_cmap(i))  # tab20 has 20 distinct
        else:
            # pick from hsv evenly for any extras
            colors.append(extra_cmap((i-10) / max(1, n_assets-10)))

    #cmap = plt.get_cmap('tab20', n_assets)
    #colors = [cmap(i) for i in range(n_assets)]

    plt.figure(figsize=(9,4))
    for i, ticker in enumerate(tickers):
        bottom = cumsum[i-1] if i>0 else np.zeros(T)
        top    = cumsum[i]
        # use dates on the x-axis
        plt.fill_between(dates, bottom, top, step='post', label=ticker, color=colors[i])

    # now the ticks are the dates themselves
    plt.xticks(
        dates[::40], 
        #[d.strftime('%Y-%m-%d') for d in dates],
        rotation=45, ha='right'
    )

    plt.xlabel('Date')
    plt.ylabel('Allocation')
    plt.title(f'Held Asset Allocations Over Time for {title}')
    plt.legend(ncol=2, bbox_to_anchor=(1.02,1), loc='upper left')
    plt.tight_layout()
    plt.show()

test_indices = df.index[-(num_test_points+10)*22:-9*22:22]
#test_indices = test_indices[:int(horizon*np.floor(num_test_points/horizon))]
#print(all_weights.shape)

def softmax_np(matrix):
    col_max = np.max(matrix, axis=0, keepdims=True)
    exp_max_reduced_values = np.exp(matrix - col_max)

    col_sums = np.sum(exp_max_reduced_values, axis=0, keepdims=True)
    softmaxed_value = exp_max_reduced_values/col_sums
    return softmaxed_value
def expand_weights(weights, horizon, num_of_test_points, only_use_positive_weights=False):
    if only_use_positive_weights:
        weights = np.clip(weights, a_min=0.0, a_max=None)
    expanded_weights = np.repeat(weights, horizon, axis=1)
    return expanded_weights[:,:num_of_test_points+1]

def allocate_from_weights(weights, horizon, num_of_test_points):
    all_weights = np.array(weights)
    all_weights_expanded = expand_weights(all_weights, horizon, num_of_test_points, only_use_positive_weights=True)

    allocations = softmax_np(all_weights_expanded)
  
    return allocations

def calc_alloc(weights, tickers, title, horizon):
    all_weights = weights
    all_weights = np.concatenate([weights, weights[:,-1:]], axis = 1)
    print(weights.shape)

    #np.save('saved_weights\\simple_lstm.npy', all_weights)

    alloc = allocate_from_weights(all_weights, horizon=horizon, num_of_test_points=num_test_points)
    print(alloc.shape)

    alloc_diff = np.diff(alloc, axis=1)
    alloc_diff = np.concatenate([alloc_diff, np.zeros((len(tickers),1))], axis=1)
    alloc_diff = 0.5* np.sum(np.abs(alloc_diff), axis=0)



    # call with your alloc, tickers and test_indices as dates:
    plot_allocations(alloc, tickers, test_indices, title)
    return alloc, alloc_diff
#np.save('best_allocations.npy',alloc)

#asGRU, adsGRU = calc_alloc(np.load('saved_weights\simple_gru.npy'), all_tickers)
asLSTM, adsLSTM = calc_alloc(np.load('saved_weights\simple_lstm_20horizon_120hidden_new_norm_40ws_5step_17_04_23.npy'), all_tickers, title='Simple LSTM', horizon=20)
#acrossGRU, adcrossGRU = calc_alloc(np.load('saved_weights\cross_gru.npy'), all_tickers)
acrossLSTM, adcrossLSTM = calc_alloc(np.load('saved_weights\cross_lstm_20horizon_120hidden_new_norm_40ws_5step_17_04_23.npy'), all_tickers, title='Cross Attention LSTM', horizon=20)

In [ ]:
from allocation_functions.allocation_functions import calculate_asset_returns_cont, calculate_portfolio_returns, calculate_cumulative_portfolio_returns
import matplotlib.pyplot as plt

def sharpe_ratio(returns, rf_ret):
    num_test_points = len(returns)
    rf_partial = rf_ret/num_test_points
    excess_ret = returns - rf_partial

    sharpe_ratio = np.sqrt(num_test_points) * excess_ret.mean() / excess_ret.std()
    return sharpe_ratio

def max_drawdown(returns):
    cumulative = (returns + 1).cumprod()
    running_max = cumulative.max()
    drawdown = cumulative/ running_max - 1
    max_drawdown = drawdown.min()
    return max_drawdown

def retrieve_test_closing_price(df, chosen_ticker, num_test_points):
    df = df[df["Ticker"] == chosen_ticker]
    df = df.iloc[20:]

    prices = df["Close"].values  
    test_prices = prices[-num_test_points-11:-9]
    return test_prices  

def calculate_asset_returns_cont(df, tickers, num_of_test_points):
    #print(num_of_test_points)
    returns = np.zeros((len(tickers), num_of_test_points+1))
    for i, ticker in enumerate(tickers):
        closing_prices = retrieve_test_closing_price(df, ticker, num_of_test_points)
        #print(closing_prices)
        #for t in range(num_of_test_points):
        #old_price = closing_prices[t]
        #new_price = closing_prices[t+1]
        current_ret = np.diff(closing_prices, axis = 0) / closing_prices[:-1]
       
        
        returns[i, :] = current_ret
                 
    return returns


def calc_final_returns(alloc, alloc_diff, tickers, model_name):
    returns = calculate_asset_returns_cont(df, tickers, num_of_test_points=num_test_points)
    portfolio_returns = calculate_portfolio_returns(alloc, returns[:,:alloc.shape[1]])

    portfolio_returns_with_tradingcost = np.where(portfolio_returns > 0, portfolio_returns - alloc_diff*trading_cost, portfolio_returns)
    portfolio_returns_with_tradingcost = portfolio_returns - alloc_diff*trading_cost

    sharpe_rat = sharpe_ratio(portfolio_returns_with_tradingcost, rf_ret=0.0)
    max_draw = max_drawdown(portfolio_returns_with_tradingcost)

    portfolio_cum_return = calculate_cumulative_portfolio_returns(portfolio_returns_with_tradingcost)

    print(f'Portfolio final return for {model_name}: ',portfolio_cum_return[-1])

    return portfolio_cum_return, sharpe_rat, max_draw

#pcr_sGRU = calc_final_returns(asGRU, adsGRU, all_tickers, 'Simple GRU')
pcr_sLSTM, ssr, smd = calc_final_returns(asLSTM, adsLSTM, all_tickers, 'Simple LSTM')
#pcr_crossGRU = calc_final_returns(acrossGRU, adcrossGRU, all_tickers, 'Cross attn. GRU')
pcr_crossLSTM, csr, cmd = calc_final_returns(acrossLSTM, adcrossLSTM, all_tickers, 'Cross attn. LSTM')

print('Simple LSTM')
print('Sharpe ratio: ',ssr)
print('Max drawdown: ', smd)

print('Cross LSTM')
print('Sharpe ratio: ',csr)
print('Max drawdown: ', cmd)


    

df_omx22 = pd.read_csv(
    "data/MCAP_22_Index_raw.csv",
    parse_dates=["Date"],
    index_col="Date"
)
index_test_values = df_omx22['MCAP_22_Index'].iloc[-510:-9].values
index_dates = df_omx22.index[-num_test_points-11:-9]
print(index_dates)
index_diffs = index_test_values / index_test_values[0]
cum_prod_index = index_diffs
print('Index final return', cum_prod_index[-1])
#510 - 5
#for i in range(5):
#    for j in range(5):
#        id1 = 508 + i
#        id2 = 8 + j
#        index_test_values = df_omx22['MCAP_22_Index'].iloc[-id1:-id2].values
#        index_dates = df_omx22.index[-num_test_points-11:-9]
#        #print(index_dates)
#        index_diffs = index_test_values / index_test_values[0]
#        cum_prod_index = index_diffs
#        print('Index final return ', cum_prod_index[-1], ' ', id1, id2)
    

plt.figure(figsize=(9,4))
plt.plot(test_indices, cum_prod_index, label='OMX22 Index')
#plt.plot(test_indices, pcr_sGRU, label='Portfolio returns Simple GRU')
plt.plot(test_indices, pcr_sLSTM, label='Portfolio returns Simple LSTM')
#plt.plot(test_indices, pcr_crossGRU, label='Portfolio returns Cross attn. GRU')
plt.plot(test_indices, pcr_crossLSTM, label='Portfolio returns Cross attn. LSTM')

plt.title('Comparative plot between the Portfolio returns and the OMX22 Index')
plt.xlabel('Date')
plt.ylabel('Return')
plt.xticks(
        test_indices[::15], 
        rotation=45, ha='right'
    )
plt.legend()

In [ ]:
#portfolio_dates = result_crossattn_lstm["dates"]
crossattn_lstm_portfolio_vals = pcr_crossLSTM
index_vals = cum_prod_index
lstm_portfolio_vals = pcr_sLSTM
portfolio_dates = test_indices

#np.save('index_vals.npy',cum_prod_index)


plt.figure(figsize=(12, 4))

plt.plot(portfolio_dates[:], np.array(index_vals[:])/index_vals[0], label="OMXS22 index", color="black", linewidth=1.25, linestyle="--")
plt.plot(portfolio_dates[:], np.array(lstm_portfolio_vals[:])/lstm_portfolio_vals[0], label="LSTM returns", color="green", linewidth=1.0)
plt.plot(portfolio_dates[:], np.array(crossattn_lstm_portfolio_vals[:])/crossattn_lstm_portfolio_vals[0], label="CrossAttention LSTM returns", color="darkorange", linewidth=1.0)
plt.xlabel("Date")
plt.xticks(
portfolio_dates[::50],
#[d.strftime('%Y-%m-%d') for d in dates],
rotation=45, ha='right'
)
plt.ylabel("Return")
plt.title("Model vs. Index returns")
plt.grid()
plt.legend()
plt.show()
print(f"LSTM portfolio value: {(np.array(lstm_portfolio_vals[:])/lstm_portfolio_vals[0])[-1]:.4f}")
print(f"CrossAttention LSTM portfolio value: {(np.array(crossattn_lstm_portfolio_vals[:])/crossattn_lstm_portfolio_vals[0])[-1]:.4f}")
print(f"Index value: {(np.array(index_vals[:])/index_vals[0])[-1]:.4f}")

def sharpe_ratio(returns, risk_free_rate=0.0):
    """
    Calculate the Sharpe ratio of a series of returns.
    """
    excess_returns = returns - risk_free_rate
    return np.mean(excess_returns) / np.std(excess_returns)

def max_drawdown(returns):
    """
    Calculate the maximum drawdown of a series of returns.
    """
    peak = np.maximum.accumulate(returns)
    drawdown = (returns - peak) / (1 + peak)
    return np.min(drawdown)

# Calculate the performance metrics
lstm_returns = np.array(lstm_portfolio_vals[:])/lstm_portfolio_vals[0] - 1
crossattn_lstm_returns = np.array(crossattn_lstm_portfolio_vals[:])/crossattn_lstm_portfolio_vals[0] - 1
index_returns = np.array(index_vals[:])/index_vals[0] - 1
lstm_sharpe = sharpe_ratio(lstm_returns)
crossattn_lstm_sharpe = sharpe_ratio(crossattn_lstm_returns)
index_sharpe = sharpe_ratio(index_returns)
lstm_max_drawdown = max_drawdown(lstm_returns)
crossattn_lstm_max_drawdown = max_drawdown(crossattn_lstm_returns)
index_max_drawdown = max_drawdown(index_returns)
print(f"LSTM Sharpe Ratio: {lstm_sharpe:.4f}")
print(f"CrossAttention LSTM Sharpe Ratio: {crossattn_lstm_sharpe:.4f}")
print(f"Index Sharpe Ratio: {index_sharpe:.4f}")
print(f"LSTM Max Drawdown: {lstm_max_drawdown:.4f}")
print(f"CrossAttention LSTM Max Drawdown: {crossattn_lstm_max_drawdown:.4f}")
print(f"Index Max Drawdown: {index_max_drawdown:.4f}")